In [ ]:
import pandas as pd
from pathlib import Path

MODEL_INPUTS_PATH = Path(r"C:\dsarp_outputs\logging_log4j2_model_inputs_from_csv.csv")

model_inputs = pd.read_csv(MODEL_INPUTS_PATH)

print("Rows:", len(model_inputs))
print("Columns:", model_inputs.columns.tolist())

display(model_inputs[["architecture_smell", "input_text"]].head())

In [ ]:
# lOADING THE TRAINED MODEL
import json
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

FINAL_MODEL_DIR = Path(
    r"C:\dsarp_outputs\models\distilbert_improved_strict_ranked_top5\final_model"
)

loaded_tokenizer = AutoTokenizer.from_pretrained(str(FINAL_MODEL_DIR))
loaded_model = AutoModelForSequenceClassification.from_pretrained(str(FINAL_MODEL_DIR))

with open(FINAL_MODEL_DIR / "labels.json", "r", encoding="utf-8") as f:
    label_data = json.load(f)

id2label = {int(k): v for k, v in label_data["id2label"].items()}
MAX_LENGTH = label_data.get("max_length", 256)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
loaded_model.to(device)
loaded_model.eval()

print("Loaded model:", FINAL_MODEL_DIR)
print("Device:", device)

In [ ]:
# DEFINING SUGGESTIONS
def suggest_refactorings(input_text, top_k=5):
    encoded = loaded_tokenizer(
        input_text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=MAX_LENGTH,
    )

    encoded = {
        key: value.to(device)
        for key, value in encoded.items()
    }

    with torch.no_grad():
        outputs = loaded_model(**encoded)
        probabilities = torch.sigmoid(outputs.logits[0]).cpu().numpy()

    top_indices = probabilities.argsort()[-top_k:][::-1]

    return [
        {
            "label": id2label[int(i)],
            "score": float(probabilities[i]),
        }
        for i in top_indices
    ]

In [ ]:
#Testing one row
sample_input = model_inputs.iloc[0]["input_text"]

print(sample_input)

suggest_refactorings(sample_input, top_k=5)

In [ ]:
# generate suggestions for all rows
suggestion_rows = []

for _, row in model_inputs.iterrows():
    suggestions = suggest_refactorings(row["input_text"], top_k=5)

    suggestion_rows.append({
        "project": row.get("project", ""),
        "versionId": row.get("versionId", ""),
        "architecture_smell": row.get("architecture_smell", ""),
        "affected_elements": row.get("affected_elements", ""),
        "input_text": row["input_text"],
        "suggestions": " | ".join(
            f"{item['label']} ({item['score']:.3f})"
            for item in suggestions
        ),
    })

suggestions_df = pd.DataFrame(suggestion_rows)

display(suggestions_df.head(20))

In [ ]:
# saving suggestions
OUTPUT_SUGGESTIONS_PATH = Path(
    r"C:\dsarp_outputs\logging_log4j2_refactoring_suggestions_from_trained_model.csv"
)

suggestions_df.to_csv(OUTPUT_SUGGESTIONS_PATH, index=False)

print("Saved suggestions to:", OUTPUT_SUGGESTIONS_PATH)